In [20]:
from alpaca.trading.client import TradingClient
from alpaca.data.historical import CryptoHistoricalDataClient
from alpaca.data.requests import CryptoBarsRequest
from alpaca.data import TimeFrame
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce
import pandas as pd
import datetime
import time

# Sustituye con tus credenciales de Paper Trading
API_KEY = "PK2CNGKUVBXK5XM75N64PIQIRL"
SECRET_KEY = "G1Ag2nzs6cFosZvBRp1QKTFLvw5nYBvNidn1W8Em9zNE"

trading_client = TradingClient(API_KEY, SECRET_KEY, paper=True)
data_client = CryptoHistoricalDataClient()

## 2. Obtención de Datos y Lógica de la Estrategia
Esta función descarga los últimos precios y calcula si debemos comprar o vender.

In [21]:
def get_signal(symbol="BTC/USD"):
    try:
        # Definimos explícitamente el FIN (hace 1 minuto, vela cerrada) y el INICIO
        end_time = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=1)
        start_time = end_time - datetime.timedelta(minutes=50)
        
        request_params = CryptoBarsRequest(
            symbol_or_symbols=[symbol],
            timeframe=TimeFrame.Minute,
            start=start_time,
            end=end_time  # <--- AÑADIMOS EXPLÍCITAMENTE EL FINAL AQUÍ
        )
        
        bars = data_client.get_crypto_bars(request_params).df
        
        # Validamos que nos hayan llegado datos para evitar errores
        if bars.empty:
            print("Esperando datos del mercado...")
            return "wait"
            
        df = bars.loc[symbol]
        
        # Calculamos la Media Móvil (SMA 20)
        df['sma_20'] = df['close'].rolling(window=20).mean()
        
        last_price = df['close'].iloc[-1]
        last_sma = df['sma_20'].iloc[-1]
        
        hora_actual = datetime.datetime.now().strftime('%H:%M:%S')
        print(f"[{hora_actual}] BTC: ${last_price:.2f} | SMA20: ${last_sma:.2f}")
        
        if last_price > last_sma: return "buy"
        elif last_price < last_sma: return "sell"
        return "wait"
        
    except Exception as e:
        print(f"Error obteniendo datos: {e}")
        return "wait"

## 3. Ejecución de Operaciones
Alpaca permite trading fraccional en cripto, lo cual es ideal para tu presupuesto de 1$.

In [22]:
def execute_trade(side, symbol="BTC/USD"):
    pos_symbol = symbol.replace("/", "") 
    
    # 1. OBTENER POSICIÓN ACTUAL
    positions = trading_client.get_all_positions()
    btc_position = next((p for p in positions if p.symbol == pos_symbol), None)

    # 2. REPORTE DE ESTADO Y GESTIÓN DE RIESGO (SL/TP)
    if btc_position:
        profit_pct = float(btc_position.unrealized_plpc)
        print(f"📊 ESTADO: Tienes una posición abierta en {symbol} | PnL actual: {profit_pct*100:.3f}%")
        
        # Revisamos Take Profit y Stop Loss primero
        if profit_pct >= 0.005:
            trading_client.close_all_positions()
            print(f"✅ TAKE PROFIT: Posición cerrada con ganancia del {profit_pct*100:.2f}%")
            return 
            
        elif profit_pct <= -0.0025:
            trading_client.close_all_positions()
            print(f"🛑 STOP LOSS: Posición cerrada con pérdida del {profit_pct*100:.2f}%")
            return 
    else:
        print(f"👀 ESTADO: No tienes posiciones abiertas en {symbol}.")

    # 3. LÓGICA DE EJECUCIÓN SEGÚN LA SEÑAL
    if side == "buy":
        if not btc_position:
            order_data = MarketOrderRequest(
                symbol=symbol,
                notional=10.00,
                side=OrderSide.BUY,
                time_in_force=TimeInForce.GTC
            )
            trading_client.submit_order(order_data)
            print("🚀 ACCIÓN: Orden de COMPRA enviada ($10). Estrategia alcista confirmada.")
        else:
            # ¡Aquí está lo que pediste!
            print("⏳ IGNORADO: Señal de COMPRA recibida, pero YA TIENES una posición abierta. Esperando a vender...")

    elif side == "sell":
        if btc_position:
            trading_client.close_all_positions()
            print("🔄 ACCIÓN: Posición CERRADA (Venta). El precio cruzó la media hacia abajo.")
        else:
            print("⏳ IGNORADO: Señal de VENTA recibida, pero NO HAY posición para vender. Esperando oportunidad de compra...")
            
    elif side == "wait":
        if btc_position:
            print("⏸️  ACCIÓN: Ninguna. Manteniendo la posición abierta y esperando evolución...")
        else:
            print("⏸️  ACCIÓN: Ninguna. Esperando que el precio cruze la media para comprar...")

## 4. El Bucle Principal (El Bot en marcha)
Este bloque mantiene al bot escuchando el mercado cada minuto.

In [ ]:
print("--- Bot Iniciado en modo Paper Trading ---")

while True:
    try:
        signal = get_signal()
        execute_trade(signal)
    except Exception as e:
        print(f"Error: {e}")
    
    # Esperamos 60 segundos para la siguiente vela
    time.sleep(60)

--- Bot Iniciado en modo Paper Trading ---
[10:39:16] BTC: $69653.62 | SMA20: $69615.91
👀 ESTADO: No tienes posiciones abiertas en BTC/USD.
🚀 ACCIÓN: Orden de COMPRA enviada ($10). Estrategia alcista confirmada.
[10:40:17] BTC: $69686.59 | SMA20: $69614.49
📊 ESTADO: Tienes una posición abierta en BTC/USD | PnL actual: -0.009%
⏳ IGNORADO: Señal de COMPRA recibida, pero YA TIENES una posición abierta. Esperando a vender...
[10:41:17] BTC: $69663.35 | SMA20: $69614.20
📊 ESTADO: Tienes una posición abierta en BTC/USD | PnL actual: -0.009%
⏳ IGNORADO: Señal de COMPRA recibida, pero YA TIENES una posición abierta. Esperando a vender...
[10:42:17] BTC: $69663.93 | SMA20: $69614.43
📊 ESTADO: Tienes una posición abierta en BTC/USD | PnL actual: -0.009%
⏳ IGNORADO: Señal de COMPRA recibida, pero YA TIENES una posición abierta. Esperando a vender...
[10:43:18] BTC: $69668.45 | SMA20: $69614.91
📊 ESTADO: Tienes una posición abierta en BTC/USD | PnL actual: -0.009%
⏳ IGNORADO: Señal de COMPRA recib